# 🟢 Cell 1: 환경 설정 및 동기화
드라이브 마운트, 깃허브 최신화, 라이브러리 설치 및 허깅페이스 로그인을 담당

In [1]:
import os
import sys
from google.colab import drive, userdata
from huggingface_hub import login

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 깃허브 동기화 및 브랜치 체크아웃
REPO_NAME = "MultiLexNorm_HW11"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = "jaeiko"
GIT_REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

print("\n--- 깃허브 저장소 동기화 ---")
if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {GIT_REPO_URL}
    print(f"[System] {REPO_NAME} 클론 완료.")
    !cd /content/{REPO_NAME} && git checkout detection-model
else:
    print(f"[System] {REPO_NAME} 폴더 존재. detection-model 최신 코드를 pull 합니다.")
    !cd /content/{REPO_NAME} && git checkout detection-model && git pull origin detection-model

# 3. 작업 디렉토리 이동 및 경로 고정
repo_path = f'/content/{REPO_NAME}'
os.chdir(repo_path)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 4. 필수 라이브러리 설치 및 호환성 에러 해결
with open('requirements.txt', 'r') as f:
    reqs = f.read()
reqs = reqs.replace('pandas<2.0', 'pandas').replace('numpy<2.0', 'numpy').replace('pyarrow==14.0.2', 'pyarrow')
with open('requirements.txt', 'w') as f:
    f.write(reqs)

!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate

# 허깅페이스 모델 다운로드 경로를 구글 드라이브 내부로 강제 지정
os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_CACHE'

# 5. 허깅페이스 로그인
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("[System] 환경 설정 및 허깅페이스 로그인 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- 깃허브 저장소 동기화 ---
[System] MultiLexNorm_HW11 폴더 존재. detection-model 최신 코드를 pull 합니다.
M	requirements.txt
Already on 'detection-model'
Your branch is up to date with 'origin/detection-model'.
From https://github.com/jaeiko/MultiLexNorm_HW11
 * branch            detection-model -> FETCH_HEAD
Already up to date.
[System] 환경 설정 및 허깅페이스 로그인 완료!


# 🟡 Cell 2: 파이프라인 엔진 및 17개국 프롬프트 정의
캐싱 문제를 완벽하게 피하고 직관적으로 관리할 수 있도록, 17개국 프롬프트가 적용된 LlamaCorrector와 메인 파이프라인 클래스를 여기에 정의합니다.

In [2]:
import glob
import ast
import json
import warnings
import pandas as pd
from typing import List
from tqdm.auto import tqdm
import importlib

import llm
importlib.reload(llm) # 깃허브에서 가져온 최신 llm.py 강제 로드

# [핵심] 코랩에 길게 쓰지 않고, 모듈에서 깔끔하게 임포트합니다.
from llm import MultilingualCorrector
from detection import AnomalyDetector
from dictionary import MFRDictionary

warnings.filterwarnings('ignore')

class LexicalNormalizationPipeline:
    def __init__(self, detector, dictionary, llm):
        self.detector = detector
        self.dictionary = dictionary
        self.llm = llm

    def process_batch(self, batch_tokens: List[List[str]], gemma_flags: List[int], batch_langs: List[str]) -> List[List[str]]:
            all_corrected_sentences = []

            # zip으로 묶을 때 lang 변수도 같이 뽑아낸다.
            for tokens, g_flag, lang in zip(batch_tokens, gemma_flags, batch_langs):
                xlmr_flags = self.detector.predict(tokens)

                if g_flag == 1 and sum(xlmr_flags) == 0:
                    final_flags = [1] * len(tokens)
                else:
                    final_flags = xlmr_flags

                corrected_sentence = []
                for idx, token in enumerate(tokens):
                    if final_flags[idx] == 1:
                        # 사전 검색 시 언어 코드(lang)를 반드시 함께 넘겨준다.
                        dict_result = self.dictionary.correct_if_confident(
                            token,
                            lang,
                            min_confidence=0.75,
                            min_count=2,
                            min_margin=0.50,
                        )

                        # 빈 문자열("")도 정답으로 통과시킨다.
                        if dict_result is not None:
                            corrected_sentence.append(dict_result)
                        else:
                            llm_result = self.llm.correct(token, tokens, lang=lang)
                            # 구문 확장(Phrase) 허용 방어 로직
                            if len(llm_result) > max(len(token) * 4, 15):
                                corrected_sentence.append(token)
                            else:
                                corrected_sentence.append(llm_result)
                    else:
                        corrected_sentence.append(token)
                all_corrected_sentences.append(corrected_sentence)

            return all_corrected_sentences

def load_gemma_sentence_predictions(flat_pred_path: str) -> list:
    with open(flat_pred_path, 'r', encoding='utf-8') as f:
        return [int(line.strip()) for line in f if line.strip() in ['0', '1']]

print("[System] 언어 인식(Lang-aware) 파이프라인 모듈 정의 완료!")

[System] 언어 인식(Lang-aware) 파이프라인 모듈 정의 완료!


# 🔴 Cell 3: 전체 데이터셋 전수 검사 및 평가
이 셀을 실행하면 임시 사전을 만들고, 8,400개 데이터 전체를 돌며 결과를 백업하고 최종 스코어를 산출합니다.

In [5]:
import glob
import ast
import json
import pandas as pd
from tqdm.auto import tqdm

print("\n--- 🚀 다국어 사전 연동 파이프라인 (전체 검증 데이터 전수 검사) ---")

# 1. 모델 조립 (탐지 모델, MFR 사전, LLM 교정 모듈)
model_path = "/content/drive/MyDrive/MultiLexNorm2026/models/xlmr_finetuned_colab"
test_detector = AnomalyDetector(model_path)

DICT_PATH = "/content/mfr_dictionary.json"
test_dictionary = MFRDictionary(DICT_PATH)
test_llm = MultilingualCorrector("Qwen/Qwen2.5-7B-Instruct")

pipeline = LexicalNormalizationPipeline(test_detector, test_dictionary, test_llm)

# 2. 데이터 로드
search_pattern = "/content/MultiLexNorm_HW11/*/LLM-base*/experiments/exp_009_retrieval_fewshot_v1_full_all_k20/predictions.txt"
found_paths = glob.glob(search_pattern)
gemma_flags_list = load_gemma_sentence_predictions(found_paths[0])

val_search_pattern = "/content/MultiLexNorm_HW11/multilexnorm2026-dataset/data/validation-00000-of-00001.parquet"
val_found_paths = glob.glob(val_search_pattern)
val_df = pd.read_parquet(val_found_paths[0])

# [핵심 변경] 데이터프레임의 전체 길이로 확장
total_samples = len(val_df)
print(f"[System] 총 {total_samples}개의 문장 추론을 본격적으로 시작합니다...")

# 3. 배치 추론 및 평가 루프
BATCH_SIZE = 8
final_predictions = []
correct_tokens_count = 0
total_tokens_count = 0

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Batch Inferencing"):
    batch_idx = range(i, min(i + BATCH_SIZE, total_samples))

    batch_raw = []
    batch_norm = []
    batch_gemma = []
    batch_langs = []

    for idx in batch_idx:
        raw = val_df['raw'].iloc[idx]
        if isinstance(raw, str): raw = ast.literal_eval(raw)
        batch_raw.append([str(t) for t in raw])

        norm = val_df['norm'].iloc[idx]
        if isinstance(norm, str): norm = ast.literal_eval(norm)
        batch_norm.append([str(t) for t in norm])

        batch_gemma.append(gemma_flags_list[idx])

        lang_code = val_df['language'].iloc[idx] if 'language' in val_df.columns else val_df['lang'].iloc[idx]
        batch_langs.append(str(lang_code))

    # 파이프라인 처리
    results = pipeline.process_batch(batch_raw, batch_gemma, batch_langs)
    final_predictions.extend(results)

    # 실시간 점수 누적
    for res, norm_ans in zip(results, batch_norm):
        total_tokens_count += len(norm_ans)
        correct_tokens_count += sum([1 for p, n in zip(res, norm_ans) if p == n])

    # [안정성 확보] 500개 단위 중간 백업
    if len(final_predictions) % 500 < BATCH_SIZE:
        with open("predictions_full_backup.json", "w", encoding="utf-8") as f:
            json.dump(final_predictions, f, ensure_ascii=False)

# 4. 최종 결과 저장
with open("predictions_validation_full.json", "w", encoding="utf-8") as f:
    json.dump(final_predictions, f, ensure_ascii=False)

accuracy = (correct_tokens_count / total_tokens_count) * 100
print(f"\n🎉 [전수 검사 완료] 전체 데이터 추론이 완료되었으며 'predictions_validation_full.json'에 저장되었습니다.")
print(f"📊 최종 단어 단위 일치율(Token Accuracy): {accuracy:.2f}%")


--- 🚀 다국어 사전 연동 파이프라인 (전체 검증 데이터 전수 검사) ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[System] Qwen/Qwen2.5-7B-Instruct 모델을 4-bit로 로드합니다. (안전망 가동)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ERR = (TP - FP) / (TP + FN)
TP = FP = FN = 0
for pred, norm, raw in zip(final_predictions, val_df['norm'], val_df['raw']):
    raw = [str(t) for t in (ast.literal_eval(raw) if isinstance(raw, str) else raw)]
    norm = [str(t) for t in (ast.literal_eval(norm) if isinstance(norm, str) else norm)]
    for p, n, r in zip(pred, norm, raw):
        TP += int(r != n and p == n); FN += int(r != n and p != n); FP += int(r == n and p != r)
ERR = (TP - FP) / (TP + FN) if (TP + FN) else 0
print(f"ERR: {ERR:.4f} ({ERR * 100:.2f}%) | TP={TP}, FP={FP}, FN={FN}")
